# Installazione librerie

In [ ]:
!pip install gradio

In [ ]:
!pip install -q transformers datasets accelerate torchvision wandb

In [ ]:
!pip install smolagents transformers accelerate diffusers pyyaml pillow

# System Prompt

In [ ]:
%%writefile prompts.yaml
# =========================================================
# I. SYSTEM PROMPT: The Cognitive Architecture
# =========================================================
system_prompt: |-
  You are an expert AI Orchestrator specialized in Computational Pathology.
  Your mission is to act as a high-level decision support system for human pathologists, orchestrating the analysis of histological specimens.

  You operate within a CodeAgent architecture. You will be given a task regarding a histological image to solve as best you can. To do so, you have been given access to specific tools, which are Python functions you can call using code.

  To solve the task, you must proceed in a series of steps, in a strict cycle of Thought, Code, and Observation sequences.

  At each step:
  1. In the 'Thought:' sequence, explain your clinical reasoning and state which tools you intend to use.
  2. In the 'Code:' sequence, write simple Python code. The code sequence MUST be opened with '{{code_block_opening_tag}}' and closed with '{{code_block_closing_tag}}'.

  =========================================
  TOOL INJECTION (Dynamic Interface)
  =========================================
  Here are the tools available to you. They behave like regular Python functions:

  {{code_block_opening_tag}}
  {%- for tool in tools.values() %}
  {{ tool.to_code_prompt() }}
  {% endfor %}
  {{code_block_closing_tag}}

  {%- if managed_agents and managed_agents.values() | list %}
  You can also give tasks to team members (managed agents):
  {{code_block_opening_tag}}
  {%- for agent in managed_agents.values() %}
  def {{ agent.name }}(task: str, additional_args: dict) -> str:
      """{{ agent.description }}"""
  {% endfor %}
  {{code_block_closing_tag}}
  {%- endif %}

  =========================================
  MANDATORY CLINICAL PROTOCOLS
  =========================================
  1. VISUAL BLINDNESS: You cannot see images directly. You MUST call the 'analizza_vetrino_istologico' tool to extract morphological data from the provided image path.
  2. QUESTION ANSWERING & DEDUCTIVE REASONING: Your primary goal is to DIRECTLY ANSWER the user's specific question. You are authorized to use your medical knowledge to interpret the tool's output (e.g., deducing that a carcinoma implies a neoplasm). However, NEVER invent visual details, cells, or structures that the tool did not explicitly detect.
  3. CLINICAL CAUTION: You are an AI assistant, not a doctor. Formulate your answer using professional, probabilistic English language.
  4. ETHICAL BOUNDARY: The final clinical responsibility rests solely with the human medical team.

  =========================================
  STRICT CODING RULES (CodeAgent Engine)
  =========================================
  1. ALWAYS provide a 'Thought:' sequence followed by a code block using '{{code_block_opening_tag}}' and '{{code_block_closing_tag}}'. If you use Markdown backticks (```), the system will fail.
  2. AVOID 'Out: None': To ensure the output of your code is captured in the Observation logs, the LAST LINE of your code block must be the variable name or the function call itself. DO NOT use `print()` on the last line.
  3. 'final_answer' IS A TOOL: NEVER assign it as a variable. ALWAYS call it as a function.
  4. NO REPETITION: Call a tool only when needed. If you already have the visual description in the logs, proceed immediately to synthesizing the final answer.
  5. You can use imports in your code, but only from the following list of authorized modules: {{authorized_imports}}
  6. The state persists between code executions, meaning variables defined in Step 1 are available in Step 2.

  {%- if custom_instructions %}
  {{custom_instructions}}
  {%- endif %}

  Now Begin!

# =========================================================
# II. PLANNING (The Cognitive Roadmap)
# =========================================================
planning:
  initial_facts: |-
    ### 1. Facts given in the task
    The image specimen to analyze is "{{task}}".
    ### 2. Facts to look up
    Morphological features using the 'analizza_vetrino_istologico' tool.
    ### 3. Facts to derive
    Clinical synthesis and consistency with known histopathological patterns.
  initial_plan: |-
    1. Invoke 'analizza_vetrino_istologico' to obtain the technical caption from the visual model.
    2. Analyze the 'Observation' log and correlate it with the user's specific question.
    3. Directly answer the user's question by combining the visual evidence with your medical reasoning.
    4. Call 'final_answer' with your targeted response and the mandatory disclaimer.
    <end_plan>
  update_facts_pre_messages: |-
    Gathering updated facts based on the execution history...
  update_facts_post_messages: |-
    ### 1. Facts inherited from task
    ### 2. Facts learned through tool execution
  update_plan_pre_messages: |-
    Evaluating the next steps for the histopathological analysis...
  update_plan_post_messages: |-
    Finalizing the synthesis and preparing for task termination.
    <end_plan>

# =========================================================
# III. MANAGED AGENT (Team Coordination)
# =========================================================
managed_agent:
  task: |-
    You are a helpful agent named '{{name}}'. Task: {{task}}
  report: |-
    Final answer from '{{name}}': {{final_answer}}

# =========================================================
# IV. FINAL ANSWER (The Clinical Output)
# =========================================================
final_answer:
  pre_messages: |-
    You are generating the final histological assessment. Ensure the tone is academic, cautious, and structured.
  post_messages: |-
    Task finalized. Protocol completed.
  render: |-
    ### Histopathological Assessment Report

    **Morphological Observations:**
    {{final_answer}}

    ---
    **PROFESSIONAL NOTICE:** This analysis is generated by an Artificial Intelligence system (BLIP-QWEN) for research support only. It does not constitute a definitive medical diagnosis. Final clinical interpretation must be rendered by a certified human pathologist.

# Parametri

In [ ]:
%%writefile pipeline_config.json
{
  "model_paths": {
    "blip2_vision_path": "giulia2005/BLIP2-Istopatologia-Tirocinio",
    "qwen_llm_id": "Qwen/Qwen2.5-Coder-32B-Instruct"
  },
  "blip2_generation_params": {
    "max_new_tokens": 40,
    "temperature": 0.15,
    "top_p": 0.85,
    "repetition_penalty": 1.2,
    "no_repeat_ngram_size": 2,
    "prompt": "Morphological description of the histological structures observed in the tissue: "
  },
  "qwen_llm_params": {
    "max_tokens": 2096,
    "temperature": 0.5
  }
}

# Agente

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool, FinalAnswerTool
import yaml
import os
import json
import torch
import warnings
import transformers
from PIL import Image
from transformers import Blip2ForConditionalGeneration, AutoProcessor

# ==========================================
# 0. LOADING CONFIGURATION
# ==========================================
try:
    with open('pipeline_config.json', 'r') as f:
        config = json.load(f)
    print("Pipeline configuration loaded successfully!")
except FileNotFoundError:
    raise RuntimeError("ERROR: pipeline_config.json file not found.")

# ==========================================
# 1. SETUP and LOG CLEANING 
# ==========================================
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
# Insert Hugging Face access token here
os.environ["HF_TOKEN"] = "INSERT_HUGGING_FACE_ACCESS_TOKEN"

# ==========================================
# 2. UPLOADING MODELS (Using Config)
# ==========================================
# A. Qwen 32B
model_llm = InferenceClientModel(
    max_tokens=config["qwen_llm_params"]["max_tokens"],
    temperature=config["qwen_llm_params"]["temperature"],
    model_id=config["model_paths"]["qwen_llm_id"],
)

final_answer = FinalAnswerTool()

# B. BLIP-2
print("Loading visual technician in progress... ")
path_modello = config["model_paths"]["blip2_vision_path"]
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(path_modello)
model_vision = Blip2ForConditionalGeneration.from_pretrained(
    path_modello,
    torch_dtype=torch.float16,
    device_map={"": 0} if device == "cuda" else None
)
model_vision.eval()
print("Visual model ready in VRAM!\n")

# ==========================================
# 3. TOOL (Whit Error Handling)
# ==========================================
@tool
def analizza_vetrino_istologico(path_immagine: str) -> str:
    """
    Performs morphological analysis of a histopathology image.
    This tool is the only way to 'see' the image. Returns a technical description in English.

    Args:
        path_immagine: The exact path of the image file (e.g., 'prova2.jpg').
    """
    import os
    from PIL import Image
    import torch

    # 1. File existence check
    path_pulito = path_immagine.strip(" '\"")
    if not path_pulito.startswith("/content/"):
        path_assoluto = os.path.join("/content/", path_pulito)
    else:
        path_assoluto = path_pulito

    if not os.path.exists(path_assoluto):
        return f"System Error: File not found at {path_assoluto}. Tell the user to check the image path."

    # 2. Inference with parameters from Config
    try:
        immagine = Image.open(path_assoluto).convert('RGB')

        p_gen = config["blip2_generation_params"]
        prompt = p_gen["prompt"]

        inputs = processor(images=immagine, text=prompt, return_tensors="pt").to(device, torch.float16)

        with torch.no_grad():
            generated_ids = model_vision.generate(
                **inputs,
                max_new_tokens=p_gen["max_new_tokens"],
                do_sample=True,
                top_p=p_gen["top_p"],
                temperature=p_gen["temperature"],
                repetition_penalty=p_gen["repetition_penalty"],
                no_repeat_ngram_size=p_gen["no_repeat_ngram_size"]
            )

        testo_pieno = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        testo_generato = testo_pieno.replace(prompt, "").strip()

        # 3. Empty output or Syntactic Loop Check
        if not testo_generato:
            return "System Error: The visual model generated an empty string. The slide might be unreadable."

        parole = testo_generato.split()
        if len(parole) > 5 and len(set(parole)) / len(parole) < 0.3:
            return f"System Error: Repetitive loop detected in visual model. Partial output: {testo_generato[:30]}..."

        return testo_generato

    except Exception as e:
        return f"System Error during visual analysis: {str(e)}. Tell the user the image processing failed."

# ==========================================
# 4. AGENT STARTUP
# ==========================================
with open("prompts.yaml", 'r') as stream:
    prompt_templates = yaml.safe_load(stream)

agent = CodeAgent(
    model=model_llm,
    tools=[analizza_vetrino_istologico, final_answer],
    max_steps=6,
    verbosity_level=1,
    prompt_templates=prompt_templates
)

# ==========================================
# 5. GRAPHICAL INTERFACE (GRADIO WEB APP)
# ==========================================
import gradio as gr

def esegui_agente(percorso_immagine, domanda_medico):
    """
    Bridge function between the Gradio interface and the Agent.
    """
    # User input error handling
    if not percorso_immagine:
        return "Error: Please upload a slide image."
    if not domanda_medico:
        return "Error: Please enter a clinical request."

    # Dynamic prompt construction
    prompt_dinamico = f"""
    Analyze the image '{percorso_immagine}'.

    User's specific clinical request: {domanda_medico}

    Task instructions:
    1. Explain your clinical reasoning step-by-step based ONLY on the morphological features found by the visual tool. Do not hallucinate features.
    2. Answer the user's specific request using your medical knowledge.
    3. Format your final answer as a professional medical report.
    """

    try:
       # Run the agent
        risultato_finale = agent.run(prompt_dinamico)

        # Adding mandatory disclaimer 
        avviso_medico = (
            "\n\n---\n"
            "**DISCLAIMER** : This analysis is generated by an Artificial Intelligence system (BLIP-QWEN). "
            "It does not constitute a definitive medical diagnosis. Final clinical interpretation must be rendered by a certified human pathologist."
        )

        return str(risultato_finale) + avviso_medico
    except Exception as e:
        return f"A critical error occurred in the Agent: {str(e)}"

# Creation of the aesthetic interface
interfaccia = gr.Interface(
    fn=esegui_agente,
    inputs=[
        gr.Image(type="filepath", label="Upload Histopathology Slide"),
        gr.Textbox(lines=2, placeholder="e.g., What are the most likely differential diagnoses?", label="Clinical Request")
    ],
    outputs=gr.Textbox(lines=15, label="Official Pathology Report:"),
    title="Multi-Agent Histopathology System",
    description="**End-to-End Prototype.**\nThe Visual Agent (BLIP-2) analyzes pixels and extracts morphology. The Clinical Agent (Qwen-32B) reasons over the extracted features and drafts the report.",
    theme="soft",
    allow_flagging="never"
)

if __name__ == "__main__":
    print("Starting the web application...")
    interfaccia.launch(share=True, debug=True)